In [0]:
# bootstrap_servers='pkc-xrnwx.asia-south2.gcp.confluent.cloud:9092'
# api_key='KUBSHIVDEZU3HTFL'
# api_secret='cflt65G+hyix5R3bmWNbnt2XKuleDlwu3Fl6+gRvAr66h1yi668oSMSk5YxWgB4w'
# topic_name='credit_card_transaction'

In [0]:
import json

In [0]:
kafka_connection_json=dbutils.secrets.get(scope="finguard-scope",key="kafka_connection_details")
kafka_config=json.loads(kafka_connection_json)
bootstrap_servers=kafka_config['bootstrap_servers']
api_key=kafka_config['api_key']
api_secret=kafka_config['api_secret']
topic_name=kafka_config['topic']    


In [0]:
jaas_config=f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'

In [0]:
sample_batch=(
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", topic_name)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("startingOffsets", "earliest")
    .load()
)

In [0]:
from pyspark.sql.functions import col

parshedBatch = sample_batch.select(
    col("key").cast("string"),
    col("value").cast("string"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")
)


In [0]:
parshedBatch.write.saveAsTable("fineguard.bronze.transactions_batch_test")

In [0]:
streaming_df=(
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", topic_name)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("startingOffsets", "earliest")
    .load()
)

In [0]:
from pyspark.sql.functions import col

parshedSreamingDf = streaming_df.select(
    col("key").cast("string"),
    col("value").cast("string"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")
)

In [0]:
streaming_query=(parshedSreamingDf.writeStream.format("delta")
 .option("checkpointLocation", "/Volumes/fineguard/source/transactions/checkpoint/")
 .trigger(availableNow=True)
 .toTable("fineguard.bronze.transactions_streaming_test")
)

print("streaming query id is :",streaming_query.id)